# TTS with emotion detetction

In [10]:
import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel
from transformers import pipeline

# 2. Initialize Emotion Classifier
# This automatically labels text as 'joy', 'anger', 'sadness', etc.
classifier = pipeline("text-classification", 
                      model="j-hartmann/emotion-english-distilroberta-base",
                      framework="pt")

# Force float32 to avoid NaN errors on Mac
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice",
    device_map="cpu", # Moving to CPU fixes the MPS multinomial crash
    torch_dtype=torch.float32, 
    trust_remote_code=True
)

def speak_with_detected_emotion(text):
    # Step A: Identify emotion from text
    detection = classifier(text)[0]
    emotion = detection['label']
    
    # Step B: Map emotion to Qwen natural language instructions
    emotion_map = {
        "joy": "Speak with a very happy, energetic, and bright tone.",
        "anger": "Speak with a loud, aggressive, and furious tone.",
        "sadness": "Speak with a slow, low-pitched, and tearful tone.",
        "fear": "Speak with a shaky, high-pitched, and panicked tone.",
        "neutral": "Speak with a calm and professional tone."
    }
    
    instruction = emotion_map.get(emotion, emotion_map["neutral"])
    print(f"Detected Emotion: {emotion} -> Using Instruction: {instruction}")

    # Step C: Generate Speech
    # Use 'Aiden' for a male voice or 'Serena' for a female voice
    wavs, sr = model.generate_custom_voice(
        text=text,
        language="English",
        speaker="Aiden", 
        instruct=instruction,
        flash_attn=False # Essential for Mac compatibility
    )

    # Step D: Save
    out_file = f"output_{emotion}.wav"
    sf.write(out_file, wavs[0], sr)
    return out_file

# Example usage:
speak_with_detected_emotion("It's finally over. I walked through the house one last time, looking at the empty rooms where we used to laugh. I guess some things just aren't meant to last forever.")

Device set to use mps:0
Ignored error while writing commit hash to /Users/grluser/.cache/huggingface/hub/models--Qwen--Qwen3-TTS-12Hz-0.6B-CustomVoice/refs/main: [Errno 28] No space left on device: '/Users/grluser/.cache/huggingface/hub/models--Qwen--Qwen3-TTS-12Hz-0.6B-CustomVoice/refs/main'.
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 61680.94it/s]
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Detected Emotion: neutral -> Using Instruction: Speak with a calm and professional tone.


'output_neutral.wav'

# With voice cloning

In [12]:
import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel
from transformers import pipeline

In [13]:
# 1. Initialize Emotion Classifier (Same as before)
classifier = pipeline("text-classification", 
                      model="j-hartmann/emotion-english-distilroberta-base",
                      framework="pt")

# 2. Load the BASE model for cloning (not CustomVoice)
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-0.6B-Base", # Base model allows cloning
    device_map="cpu", 
    torch_dtype=torch.float32, 
    trust_remote_code=True
)

Device set to use mps:0
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 20841.26it/s]


In [18]:
def clone_from_files(target_text, audio_path, transcript_path):
    # 1. Read transcript
    with open(transcript_path, 'r', encoding='utf-8') as f:
        reference_text = f.read().strip()
    
    # 2. Auto-detect emotion
    detection = classifier(target_text)[0]
    emotion = detection['label']
    confidence = detection['score'] # How sure the AI is (0.0 to 1.0)
    
    # --- VISIBILITY: This is the part you were missing ---
    print(f"\n🔍 ANALYSIS:")
    print(f"Detected Emotion: {emotion.upper()} ({confidence*100:.1f}% confidence)")
    
    # 3. Map to instructions
    emotion_map = {
        "joy": "Speak with a very happy, energetic, and bright tone.",
        "anger": "Speak with a loud, aggressive, and furious tone.",
        "sadness": "Speak with a slow, low-pitched, and tearful tone.",
        "fear": "Speak with a shaky, high-pitched, and panicked tone.",
        "surprise": "Speak with a shocked and high-energy tone.",
        "neutral": "Speak with a calm and professional tone."
    }
    instruction = emotion_map.get(emotion, emotion_map["neutral"])
    
    print(f"Instruction Sent to Qwen3: \"{instruction}\"")
    print(f"Processing... (Running on CPU may take a moment)\n")

    # 4. Generate Voice Clone
    wavs, sr = model.generate_voice_clone(
        text=target_text,
        language="English",
        ref_audio=audio_path,
        ref_text=reference_text,
        instruct=instruction,
        flash_attn=False
    )

    out_file = f"cloned_{emotion}.wav"
    sf.write(out_file, wavs[0], sr)
    print(f"✅ DONE: Generated {out_file}")

# neutral tone

In [19]:
clone_from_files(
    target_text="It's finally over. I walked through the house one last time, looking at the empty rooms where we used to laugh. I guess some things just aren't meant to last forever.", 
    audio_path="myvoice.wav", 
    transcript_path="mytranscript.txt"
)


🔍 ANALYSIS:
Detected Emotion: NEUTRAL (62.0% confidence)
Instruction Sent to Qwen3: "Speak with a calm and professional tone."
Processing... (Running on CPU may take a moment)

[WARNING] Min value of input waveform signal is -1.1195008754730225
[WARNING] Max value of input waveform signal is 1.1321690082550049


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


✅ DONE: Generated cloned_neutral.wav


In [20]:
clone_from_files(
    target_text="I told you a thousand times not to touch my things! You never listen, and I am absolutely sick of your constant excuses. Just get out and leave me alone!", 
    audio_path="myvoice.wav", 
    transcript_path="mytranscript.txt"
)


🔍 ANALYSIS:
Detected Emotion: ANGER (66.3% confidence)
Instruction Sent to Qwen3: "Speak with a loud, aggressive, and furious tone."
Processing... (Running on CPU may take a moment)

[WARNING] Min value of input waveform signal is -1.1195008754730225
[WARNING] Max value of input waveform signal is 1.1321690082550049


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


✅ DONE: Generated cloned_anger.wav
